In [122]:
from pyspark.sql.session import SparkSession

spark = (

    SparkSession
    .builder
    .appName("spark-sql-aware")
    .config("spark.sql.warehouse.dir","../data/warehouse")
    .getOrCreate()
)

spark

26/11/13 13:14:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [123]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [124]:
spark.sql("show tables").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [125]:
spark.sql("select 1 + 1").show()

+-------+
|(1 + 1)|
+-------+
|      2|
+-------+



In [126]:
spark.conf.get("spark.sql.warehouse.dir")

'file:/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/warehouse'

## Managed (Internal) vs. Unmanaged (External) Tables in Spark SQL

Spark absolutely supports both internal and external tables. In Spark terminology, these are referred to as **managed** and **unmanaged** tables. 

Every table stores two important pieces of information: the data itself, and the metadata (information about the table). 

* **Managed (Internal) Tables:** Spark manages both the metadata and the data itself. When you use the `saveAsTable` command on a DataFrame, you are creating a managed table where Spark tracks all relevant information. If you drop a managed table, you risk losing the data completely because Spark deletes the underlying files.
* **Unmanaged (External) Tables:** Spark manages the table's metadata, but the underlying data files are not managed by Spark at all. When you define a table from files already sitting on a disk, you are defining an unmanaged table. You typically create these using the `CREATE EXTERNAL TABLE` SQL statement. If you drop an external table, Spark only deletes the metadata; your data files remain untouched on the disk.

In [127]:
data = [("James", "Sales", 3000), ("Maria", "Finance", 3000)]
df = spark.createDataFrame(data, ["name", "department", "salary"])

df.show()

+-----+----------+------+
| name|department|salary|
+-----+----------+------+
|James|     Sales|  3000|
|Maria|   Finance|  3000|
+-----+----------+------+



In [128]:
## the table and data both information is managed by spark, so the data will be inside warehouse dir
df.write.mode("overwrite").saveAsTable("managed_employee")

In [129]:
spark.sql("show tables in default").show()

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|  default|managed_employee|      false|
+---------+----------------+-----------+



In [130]:
spark.sql("select * from managed_employee").show()

+-----+----------+------+
| name|department|salary|
+-----+----------+------+
|Maria|   Finance|  3000|
|James|     Sales|  3000|
+-----+----------+------+



In [131]:
spark.sql("describe formatted managed_employee").show(truncate=False)

+----------------------------+--------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                         |comment|
+----------------------------+--------------------------------------------------------------------------------------------------+-------+
|name                        |string                                                                                            |NULL   |
|department                  |string                                                                                            |NULL   |
|salary                      |bigint                                                                                            |NULL   |
|                            |                                                                                                  |       |
|# Detailed Table Information|    

In [132]:
### creating external tables

In [133]:
external_path="../tmp/spark-sql-external-tables"
df.write.mode("overwrite").option("path", external_path).saveAsTable("external_employees")

In [134]:
spark.sql("show tables").show()

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|  default|external_employees|      false|
|  default|  managed_employee|      false|
+---------+------------------+-----------+



In [135]:
spark.sql("describe formatted external_employees").show(truncate=False)

+----------------------------+-----------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                            |comment|
+----------------------------+-----------------------------------------------------------------------------------------------------+-------+
|name                        |string                                                                                               |NULL   |
|department                  |string                                                                                               |NULL   |
|salary                      |bigint                                                                                               |NULL   |
|                            |                                                                                                     |       |
|# Detailed T

In [136]:
spark.sql("DROP TABLE IF EXISTS managed_employees")
spark.sql("DROP TABLE IF EXISTS external_employees")

DataFrame[]

### loading of csv data into table 

In [137]:
! head ../data/flight-data/csv/2010-summary.csv

DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count
United States,Romania,1
United States,Ireland,264
United States,India,69
Egypt,United States,24
Equatorial Guinea,United States,1
United States,Singapore,25
United States,Grenada,54
Costa Rica,United States,477
Senegal,United States,29


In [138]:
import os

realtive_path = "../data/flight-data/csv/2010-summary.csv"
abs_path = os.path.abspath(realtive_path)

spark.sql(f"""
create table flight_csv (
          DEST_COUNTRY_NAME STRING,
          ORIGIN_COUNTRY_NAME STRING,
          count INT
          )
        USING CSV
        OPTIONS ( header true, path '{abs_path}')

""")

DataFrame[]

In [139]:
spark.sql("select * from flight_csv limit 10 ").show()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
|    United States|          Singapore|   25|
|    United States|            Grenada|   54|
|       Costa Rica|      United States|  477|
|          Senegal|      United States|   29|
|    United States|   Marshall Islands|   44|
+-----------------+-------------------+-----+



In [140]:
spark.sql("describe formatted flight_csv ").show(truncate=False)

+----------------------------+--------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                               |comment|
+----------------------------+--------------------------------------------------------------------------------------------------------+-------+
|DEST_COUNTRY_NAME           |string                                                                                                  |NULL   |
|ORIGIN_COUNTRY_NAME         |string                                                                                                  |NULL   |
|count                       |int                                                                                                     |NULL   |
|                            |                                                                                                        | 

In [141]:
spark.sql("""
create table fligt_csv_partition(
          ORIGIN_COUNTRY_NAME STRING,
          count INT,
          DEST_COUNTRY_NAME STRING
          )
          using parquet
          partitioned by (DEST_COUNTRY_NAME)
""")

spark.sql("show tables").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|  default|         flight_csv|      false|
|  default|fligt_csv_partition|      false|
|  default|   managed_employee|      false|
+---------+-------------------+-----------+



In [142]:
spark.sql("describe fligt_csv_partition").show(truncate=False)

+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|ORIGIN_COUNTRY_NAME    |string   |NULL   |
|count                  |int      |NULL   |
|DEST_COUNTRY_NAME      |string   |NULL   |
|# Partition Information|         |       |
|# col_name             |data_type|comment|
|DEST_COUNTRY_NAME      |string   |NULL   |
+-----------------------+---------+-------+



In [143]:
spark.sql("select * from fligt_csv_partition").show()

+-------------------+-----+-----------------+
|ORIGIN_COUNTRY_NAME|count|DEST_COUNTRY_NAME|
+-------------------+-----+-----------------+
+-------------------+-----+-----------------+



In [144]:
spark.sql("""select ORIGIN_COUNTRY_NAME,count 
          from flight_csv 
          where DEST_COUNTRY_NAME="United States"  limit 10
          """).show()

+--------------------+-----+
| ORIGIN_COUNTRY_NAME|count|
+--------------------+-----+
|             Romania|    1|
|             Ireland|  264|
|               India|   69|
|           Singapore|   25|
|             Grenada|   54|
|    Marshall Islands|   44|
|        Sint Maarten|   53|
|         Afghanistan|    2|
|              Russia|  156|
|Federated States ...|   48|
+--------------------+-----+



In [145]:
# inserting single partion
spark.sql("""
insert into fligt_csv_partition
          partition (DEST_COUNTRY_NAME="United States")
          select ORIGIN_COUNTRY_NAME,count 
          from flight_csv where DEST_COUNTRY_NAME="United States" limit 10

""")

DataFrame[]

In [146]:
spark.sql("msck repair table fligt_csv_partition")

DataFrame[]

In [147]:
spark.sql("select * from fligt_csv_partition").show()

+--------------------+-----+-----------------+
| ORIGIN_COUNTRY_NAME|count|DEST_COUNTRY_NAME|
+--------------------+-----+-----------------+
|             Romania|    1|    United States|
|             Ireland|  264|    United States|
|               India|   69|    United States|
|           Singapore|   25|    United States|
|             Grenada|   54|    United States|
|    Marshall Islands|   44|    United States|
|        Sint Maarten|   53|    United States|
|         Afghanistan|    2|    United States|
|              Russia|  156|    United States|
|Federated States ...|   48|    United States|
+--------------------+-----+-----------------+



In [158]:
spark.sql("describe formatted fligt_csv_partition").show(truncate=False)

+----------------------------+-----------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                            |comment|
+----------------------------+-----------------------------------------------------------------------------------------------------+-------+
|ORIGIN_COUNTRY_NAME         |string                                                                                               |NULL   |
|count                       |int                                                                                                  |NULL   |
|DEST_COUNTRY_NAME           |string                                                                                               |NULL   |
|# Partition Information     |                                                                                                     |       |
|# col_name  

In [148]:
### insert all at once

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

## select partion column at the end
spark.sql("""
insert into fligt_csv_partition
    
          select ORIGIN_COUNTRY_NAME,count, DEST_COUNTRY_NAME
          from flight_csv 
""")



DataFrame[]

In [155]:
spark.sql("""select  * from fligt_csv_partition 
          where DEST_COUNTRY_NAME != 'United States'""").show(10)

+-------------------+-----+--------------------+
|ORIGIN_COUNTRY_NAME|count|   DEST_COUNTRY_NAME|
+-------------------+-----+--------------------+
|      United States|   13|        Cook Islands|
|      United States|  995|              Brazil|
|      United States|  252|           Hong Kong|
|      United States| 1392|             Germany|
|      United States|    1|            Malaysia|
|      United States|  187| Trinidad and Tobago|
|      United States|   49|British Virgin Is...|
|      United States|   29|             Senegal|
|      United States|   65|              Sweden|
|      United States|  231|             Ireland|
+-------------------+-----+--------------------+
only showing top 10 rows


In [149]:
realtive_path = "../data/flight-data/parquet/2010-summary"
abs_path = os.path.abspath(realtive_path)

spark.sql(f"""
create table flight_parquet (
          DEST_COUNTRY_NAME STRING,
          ORIGIN_COUNTRY_NAME STRING,
          count INT
          )
        USING parquet
        OPTIONS (path '{abs_path}')

""")


DataFrame[]

In [150]:
spark.sql("describe formatted flight_parquet ").show(truncate=False)

+----------------------------+--------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                               |comment|
+----------------------------+--------------------------------------------------------------------------------------------------------+-------+
|DEST_COUNTRY_NAME           |string                                                                                                  |NULL   |
|ORIGIN_COUNTRY_NAME         |string                                                                                                  |NULL   |
|count                       |int                                                                                                     |NULL   |
|                            |                                                                                                        | 

### Catalog
The highest level abstraction in Spark SQL is the Catalog. The Catalog is an abstraction for the
storage of metadata about the data stored in your tables as well as other helpful things like
databases, tables, functions, and views. The catalog is available in the
org.apache.spark.sql.catalog.Catalog package and contains a number of helpful functions
for doing things like listing tables, databases, and functions.

In [159]:
spark.catalog.currentDatabase()

'default'

In [160]:
spark.catalog.getTable("flight_csv")

Table(name='flight_csv', catalog='spark_catalog', namespace=['default'], description=None, tableType='EXTERNAL', isTemporary=False)

### creating external tables

In [164]:
realtive_path = "..data/flight-data/csv/2010-summary.csv"
abs_path = os.path.abspath(realtive_path)

spark.sql(f"""

create external table if not exists flight_csv_external (
    DEST_COUNTRY_NAME STRING, 
    ORIGIN_COUNTRY_NAME STRING,
     count LONG
          )
    USING CSV
          OPTIONS (
          header 'true',
          delimiter ','
          )
        LOCATION '{abs_path}'
     """
)

DataFrame[]

In [168]:
spark.sql("describe formatted flight_csv_external").show(truncate=False)

+----------------------------+------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                             |comment|
+----------------------------+------------------------------------------------------------------------------------------------------+-------+
|DEST_COUNTRY_NAME           |string                                                                                                |NULL   |
|ORIGIN_COUNTRY_NAME         |string                                                                                                |NULL   |
|count                       |bigint                                                                                                |NULL   |
|                            |                                                                                                      |       |
|# Det

In [ ]:
## in case of parquet it will refere the schema from the file itself

In [170]:
realtive_path = "../data/flight-data/parquet/2010-summary"
abs_path = os.path.abspath(realtive_path)

spark.sql(f"""

create external table if not exists flight_parquet_external 
    USING CSV
          OPTIONS (
          header 'true',
          delimiter ','
          )
        LOCATION '{abs_path}'
     """
)

DataFrame[]

### loading all the csv files in one go

In [175]:
import os

# 1. Fixed the missing slash in the relative path
relative_path = "../data/flight-data/csv/"

# 2. Converted to absolute path and prepended the file URI scheme
abs_path = "file://" + os.path.abspath(relative_path)

spark.sql(f"""
    CREATE EXTERNAL TABLE IF NOT EXISTS flight_all_data (
        DEST_COUNTRY_NAME STRING, 
        ORIGIN_COUNTRY_NAME STRING,
        count BIGINT
    )
    USING CSV
    OPTIONS (
        header 'true',
        delimiter ','
    )
    LOCATION '{abs_path}'
""")

# verify it loaded correctly
spark.sql("SELECT * FROM flight_all_data LIMIT 5").show()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+



### Caching Table like dataframe

In [176]:
spark.sql("cache table flight_parquet_external ")

DataFrame[]

In [177]:
spark.catalog.isCached("flight_parquet_external")

True

In [179]:
spark.sql("uncache table flight_parquet_external ")

DataFrame[]

In [180]:
spark.catalog.isCached("flight_parquet_external")

False

Standard View (VIEW)
Scope: Bound to the database/catalog. It is accessible to anyone with access to the cluster and the specific database.

Lifespan: Persistent. The metadata is permanently saved in the Spark catalog (or Hive Metastore). It survives cluster restarts and application terminations until explicitly dropped.

Use Case: Saving complex query logic permanently so other users or downstream applications can query it tomorrow or next week.

Syntax: CREATE VIEW my_view AS SELECT ...

In [182]:
spark.sql("select distinct DEST_COUNTRY_NAME from flight_all_data").show()

+--------------------+
|   DEST_COUNTRY_NAME|
+--------------------+
|            Anguilla|
|              Russia|
|            Paraguay|
|             Senegal|
|              Sweden|
|            Kiribati|
|              Guyana|
|         Philippines|
|            Malaysia|
|           Singapore|
|                Fiji|
|              Turkey|
|             Germany|
|         Afghanistan|
|              Jordan|
|               Palau|
|Turks and Caicos ...|
|              France|
|              Greece|
|              Taiwan|
+--------------------+
only showing top 20 rows


In [199]:
spark.sql("""
create or replace view just_usa_view as 
          select * from flight_all_data where dest_country_name = 'United States'

""")

spark.sql("select count(*) from just_usa_view").show()

+--------+
|count(1)|
+--------+
|     736|
+--------+



In [196]:
spark.sql("describe formatted just_usa_view ").show(truncate=False)

+----------------------------+-----------------------------------------------------------------------+-------+
|col_name                    |data_type                                                              |comment|
+----------------------------+-----------------------------------------------------------------------+-------+
|DEST_COUNTRY_NAME           |string                                                                 |NULL   |
|ORIGIN_COUNTRY_NAME         |string                                                                 |NULL   |
|count                       |bigint                                                                 |NULL   |
|                            |                                                                       |       |
|# Detailed Table Information|                                                                       |       |
|Catalog                     |spark_catalog                                                          |       |
|

In [197]:
spark.sql("show tables ").show(truncate=False)

+---------+----------------------------+-----------+
|namespace|tableName                   |isTemporary|
+---------+----------------------------+-----------+
|default  |flight_all_data             |false      |
|default  |flight_csv                  |false      |
|default  |flight_csv_external         |false      |
|default  |flight_csv_external_all_file|false      |
|default  |flight_parquet              |false      |
|default  |flight_parquet_external     |false      |
|default  |fligt_csv_partition         |false      |
|default  |just_usa_view               |false      |
|default  |managed_employee            |false      |
|         |just_usa_view_temp          |true       |
+---------+----------------------------+-----------+



### Temp View

A temporary view in Spark SQL is a lightweight, session-scoped table-like object used for intermediate transformations.

- Scope: Bound to the specific SparkSession that created it.
- Lifespan: Disappears when the SparkSession ends or disconnects.
- Use case: Useful for quick, temporary queries inside a single script or notebook.
- Syntax:
    ```sql
    CREATE TEMP VIEW my_view AS
    SELECT ...
    ```

    or in Python:
    ```python
    df.createOrReplaceTempView("my_view")
    ```

This is especially helpful when you want to query a DataFrame using SQL without persisting it as a physical table.

In [198]:
spark.sql("""
create or replace temp view just_usa_view_temp as 
          select * from flight_all_data where dest_country_name = 'United States'

""")

spark.sql("select count(*) from just_usa_view_temp").show()

+--------+
|count(1)|
+--------+
|     736|
+--------+



In [189]:
spark.sql("show tables ").show(truncate=False)

+---------+----------------------------+-----------+
|namespace|tableName                   |isTemporary|
+---------+----------------------------+-----------+
|default  |flight_all_data             |false      |
|default  |flight_csv                  |false      |
|default  |flight_csv_external         |false      |
|default  |flight_csv_external_all_file|false      |
|default  |flight_parquet              |false      |
|default  |flight_parquet_external     |false      |
|default  |fligt_csv_partition         |false      |
|default  |just_usa_view               |false      |
|default  |managed_employee            |false      |
|         |just_usa_view_temp          |true       |
+---------+----------------------------+-----------+



### Global Temporary View (GLOBAL TEMP VIEW)

- Scope: Bound to the entire Spark application. It can be shared across multiple `SparkSession`s running within the same application.
- Lifespan: Disappears when the Spark application terminates.
- Quirk: Spark automatically places these in a system database named `global_temp`. You must explicitly prefix your queries with `global_temp.` to access them.
- Use case: Sharing data between different notebooks, threads, or users connected to the same Spark cluster/application.
- Syntax:
    ```sql
    CREATE GLOBAL TEMP VIEW my_view AS
    SELECT ...
    ```
- Querying:
    ```sql
    SELECT * FROM global_temp.my_view
    ```

These views are resolved regardless of database and are viewable across the entire Spark application, but they are removed at the end of the session.

In [193]:
spark.sql("""
create or replace global temp view just_usa_view_temp as 
select * from flight_all_data where dest_country_name = 'United States'

""")

spark.sql("select count(*) from just_usa_view_temp").show()

spark.sql("show tables ").show(truncate=False)

+--------+
|count(1)|
+--------+
|     736|
+--------+

+---------+----------------------------+-----------+
|namespace|tableName                   |isTemporary|
+---------+----------------------------+-----------+
|default  |flight_all_data             |false      |
|default  |flight_csv                  |false      |
|default  |flight_csv_external         |false      |
|default  |flight_csv_external_all_file|false      |
|default  |flight_parquet              |false      |
|default  |flight_parquet_external     |false      |
|default  |fligt_csv_partition         |false      |
|default  |just_usa_view               |false      |
|default  |managed_employee            |false      |
|         |just_usa_view_temp          |true       |
+---------+----------------------------+-----------+



In [200]:
spark.sql("""DROP VIEW IF EXISTS just_usa_view""")

DataFrame[]

### Databases



In [202]:
spark.sql("show databases""").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
+---------+



In [ ]:
spark.sql("create database some_db") # this will be create in warehouse some_db.db

DataFrame[]

In [204]:
spark.sql("show databases""").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|some_db  |
+---------+



In [205]:
spark.sql("SELECT current_database()").show()

+----------------+
|current_schema()|
+----------------+
|         default|
+----------------+



In [208]:
spark.sql("DROP DATABASE IF EXISTS some_db")

DataFrame[]

### case...when...then Statements



In [211]:
spark.sql("""

SELECT
CASE WHEN DEST_COUNTRY_NAME = 'United States' THEN 1
WHEN DEST_COUNTRY_NAME = 'Egypt' THEN 0
ELSE -1 END as flag
FROM flight_parquet

""").show()

+----+
|flag|
+----+
|   1|
|   1|
|   1|
|   0|
|  -1|
|   1|
|   1|
|  -1|
|  -1|
|   1|
|  -1|
|   1|
|  -1|
|  -1|
|  -1|
|  -1|
|   1|
|  -1|
|  -1|
|   1|
+----+
only showing top 20 rows


In [212]:
spark.stop()